In [2]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.ls_bip import laplace, lowrank
from cardiac_electrophysiology.utils import visualization

In [3]:
posterior_settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_laplace_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.05,
        tau=10,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-3,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

posterior_builder = builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
optimization_data = np.load("../results/optimization_data.npz")
map_estimate = optimization_data["map_estimate"]

In [ ]:
random_decomposer = lowrank.RandomizedDecomposer(posterior, map_estimate, seed=0)
prior_eigenvalues, prior_eigenvectors = (
    random_decomposer.compute_prior_covariance_lowrank_approximation(num_eigenvalues=1000)
)
hessian_eigenvalues, hessian_eigenvectors = (
    random_decomposer.compute_likelihood_hessian_lowrank_approximation(
        1000, scaling_factor=1, offset_factor=0
    )
)
lowrank_data = {
    "hessian_eigenvalues": hessian_eigenvalues,
    "hessian_eigenvectors": hessian_eigenvectors,
    "prior_covariance_eigenvalues": prior_eigenvalues,
    "prior_covariance_eigenvectors": prior_eigenvectors,
}
np.savez("../results/lowrank_data.npz", **lowrank_data)
visualization.plot_eigenvalues(prior_eigenvalues)
visualization.plot_eigenvalues(hessian_eigenvalues)

In [5]:
lowrank_data = np.load("../results/lowrank_data.npz")
laplace_approximation = laplace.LaplaceApproximation(
    hessian_eigenvalues=lowrank_data["hessian_eigenvalues"],
    hessian_eigenvectors=lowrank_data["hessian_eigenvectors"],
    prior_covariance_eigenvalues=lowrank_data["prior_covariance_eigenvalues"],
    prior_covariance_eigenvectors=lowrank_data["prior_covariance_eigenvectors"],
    map_estimate=map_estimate,
    prior=posterior.prior,
)

prior_variance, hessian_variance = laplace_approximation.compute_pointwise_variance()
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=prior_variance,
    circular=False,
)
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=prior_variance - hessian_variance,
    circular=False,
)

Widget(value='<iframe src="http://localhost:42943/index.html?ui=P_0x7f7500186c10_2&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:42943/index.html?ui=P_0x7f73ff15ed50_3&reconnect=auto" class="pyvi…